# AI Challenge - Prediction of Spatial Cellular Organization from Histology Images

## Random submission notebook

This notebook attempts to understand how the leaderboard is scored. It generates random submissions, then modifies a portion of the data points, to study how the Leaderbord score changes.

# 1. Data Loading and Visualization

Below, we load the HE images (both Train and Test) from the HDF5 file and overlay the spot coordinates on the images.

In [29]:
!python -m pip install -q h5py pandas matplotlib numpy kaggle


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import kaggle

In [31]:
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
api.model_list_cli()

Next Page Token = CfDJ8KT8tnOr7fFFm_byYmusL7j-WApCVd4prDdEmd_tSY9hbTETxCooQnXtGdaWrjgUrfaEC-eLB6rKNoZPRxtipAg
    id  ref                                                        title                                      subtitle                                                                                                                                                                                                                                                       author              
------  ---------------------------------------------------------  -----------------------------------------  -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------  ------------------  
184945  google/gemini-2.0-flash-api                                Gemini 2.0 Flash API                       A new fa

In [32]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()



In [33]:
# Downloads the file from the competition "el-hackathon-2025"
api.competition_download_file(
    'el-hackathon-2025', 
    'elucidata_ai_challenge_data.h5', 
    path='./data'  # downloads the file into the local folder "./data"
)


elucidata_ai_challenge_data.h5: Skipping, found more recently modified local copy (use --force to force download)


In [34]:
!dir data

 Volume in drive C is Windows
 Volume Serial Number is C2B7-260F

 Directory of c:\Users\dallo\workspace\kaggle_competitions\data

02/04/2025  11:56    <DIR>          .
05/04/2025  22:14    <DIR>          ..
12/03/2025  17:34       333,866,720 elucidata_ai_challenge_data.h5
               1 File(s)    333,866,720 bytes
               2 Dir(s)  603,394,879,488 bytes free


In [35]:
config = {
    "n_rows": 500,
    "n_cols": 25
}

In [36]:
# Display spot table for Test slide (only the spot coordinates on 2D array)
with h5py.File("./data/elucidata_ai_challenge_data.h5", "r") as f:
    test_spots = f["spots/Test"]
    spot_array = np.array(test_spots['S_7'])
    test_spot_table = pd.DataFrame(spot_array)
    
# Show the test spots coordinates for slide 'S_7'
test_spot_table

# Load training data
with h5py.File("./data/elucidata_ai_challenge_data.h5", "r") as f:
    train_spots = f["spots/Train"]
    train_spot_tables = {slide_name: pd.DataFrame(np.array(train_spots[slide_name])) for slide_name in train_spots.keys()}

In [37]:
test_spot_table.Test_Set.value_counts()


Test_Set
2    1586
1     502
Name: count, dtype: int64

In [38]:
test_spot_table

,x,y,Test_Set
0,1499,1260,2
1,1435,1503,2
2,558,1082,2
3,736,1304,1
4,1257,1592,1
...,...,...,...
2083,736,639,2
2084,1016,684,2
2085,1181,839,2
2086,735,1436,1


# 3. Creating a Random Submission

In this section, we generate random predictions for the 35 cell types.  
The predictions are random floats between 0 and 2 (without any normalization) for each spot in the Test slide.  
The order of spots is preserved as in the test spots table.

In [39]:
import time
import random
import numpy as np
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi
def generate_permuted_submission(train_spot_tables, test_spot_table, seed=None):
    """
    Generate a submission DataFrame where each row is filled with a random permutation 
    of integers from 1 to 35 (or more generally, 1 to the total number of columns).
    The submission DataFrame will include an 'ID' column corresponding to each test spot.
    The output will have one row per entry in test_spot_table (2088 rows).

    Parameters:
        train_spot_tables (dict): For getting C1–C35 column names.
        test_spot_table (pd.DataFrame): Test spot table with 2088 rows.
        seed (int, optional): Random seed for reproducibility.

    Returns:
        pd.DataFrame: Submission DataFrame with random permutations for all rows.
    """
    import numpy as np
    import pandas as pd

    if seed is not None:
        np.random.seed(seed)

    # Get column names from the training table (assumes they start at the third column)
    all_columns = train_spot_tables['S_1'].columns[2:].tolist()
    total_columns = len(all_columns)
    
    # Use the index from test_spot_table (should be 2088 rows)
    indices = test_spot_table.index
    num_rows = len(indices)
    
    # Create a matrix where each row is a random permutation of numbers from 1 to total_columns
    matrix = np.array([np.random.permutation(np.arange(1, total_columns + 1)) for _ in range(num_rows)])
    
    # Build the submission DataFrame using all rows
    submission = pd.DataFrame(matrix, index=indices, columns=all_columns, dtype=int)
    
    # Add the 'ID' column based on the index
    submission.insert(0, 'ID', submission.index)
    
    # Optionally, reset the index if you prefer a clean numeric index
    submission.reset_index(drop=True, inplace=True)
    
    return submission




In [40]:
def modify_submission(existing_submission, fraction=0.1, seed=None):
    """
    Modify an existing submission by shuffling a fraction of the prediction columns in each row.
    This ensures that each row (which should be a permutation of numbers) remains valid,
    but only a portion (e.g., 10% or 20% of the columns) are randomly modified.
    
    Parameters:
        existing_submission (pd.DataFrame): DataFrame with an 'ID' column and permutation columns.
        fraction (float): Fraction of prediction columns to modify in each row (0 < fraction < 1).
        seed (int, optional): Random seed for reproducibility.
    
    Returns:
        pd.DataFrame: Modified submission DataFrame.
    """
    import numpy as np
    import pandas as pd

    if seed is not None:
        np.random.seed(seed)
    
    # Work on a copy so the original is preserved
    modified = existing_submission.copy()
    
    # Assuming the first column is "ID", and the remaining are the prediction columns.
    perm_columns = modified.columns[1:]
    total_columns = len(perm_columns)
    
    # Determine the number of columns to modify (at least one).
    k = max(1, int(round(total_columns * fraction)))
    
    # Iterate through each row and modify a subset of its permutation values.
    for idx, row in modified.iterrows():
        # Extract current permutation for the row
        current_perm = row[perm_columns].values.copy()
        # Randomly select k column indices to modify
        indices_to_modify = np.random.choice(total_columns, size=k, replace=False)
        # Extract the subset and shuffle it
        subset = current_perm[indices_to_modify].copy()
        np.random.shuffle(subset)
        # Replace the selected positions with the shuffled values
        current_perm[indices_to_modify] = subset
        # Write the modified row back into the DataFrame
        modified.loc[idx, perm_columns] = current_perm
    
    return modified

# Example usage:
# Assume `submission_df` is your existing submission DataFrame.
# Modify 10% of the points (adjust fraction=0.2 for 20% modifications)
#modified_submission_df = modify_submission(submission_df, fraction=0.1, seed=42)
#print(modified_submission_df.head())


In [41]:


# Set the competition slug (update as needed)
competition_slug = "el-hackathon-2025"

# Number of submissions you want to generate and submit
num_submissions = 20

for i in range(num_submissions):
    # Generate a submission with an optionally randomized seed
    #submission_df = generate_permuted_submission(train_spot_tables, test_spot_table, seed=np.random.randint(10000))
    original_submission_df = pd.read_csv("submission_02970_CNN_resizing_v6.csv")
    original_submission_df = pd.read_csv("submission_patch_45.csv")

    submission_df = modify_submission(original_submission_df, fraction=0.5, seed=i+20)
    
    # Save the submission file locally
    filename = f"submission_{i}.csv"
    submission_df.to_csv(filename, index=False)
    
    submission_df.head()
    # Prepare a submission message
    submission_message = f"Automated submission {i}"
    
    # Submit the file to the competition
    api.competition_submit(file_name=filename, message=submission_message, competition=competition_slug)
    print(f"Submitted {filename} to {competition_slug}")
    
    # Wait for a random time between 3 and 5 seconds before the next submission
    wait_time = random.uniform(3, 5)
    time.sleep(wait_time)


100%|██████████| 752k/752k [00:00<00:00, 783kB/s] 


Submitted submission_0.csv to el-hackathon-2025


100%|██████████| 752k/752k [00:00<00:00, 802kB/s] 


Submitted submission_1.csv to el-hackathon-2025


100%|██████████| 752k/752k [00:01<00:00, 756kB/s] 


...

100%|██████████| 752k/752k [00:01<00:00, 748kB/s] 


Submitted submission_10.csv to el-hackathon-2025


 72%|███████▏  | 544k/752k [00:00<00:00, 1.41MB/s]2025-04-05 22:23:25,901 WARNING Retrying (Retry(total=9, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)': /upload/storage/v1/b/kaggle-competitions-submissions/o?uploadType=resumable&upload_id=AKDAyIslD5pB0lN6p7F1UdonN_6zYcdkS1NteRpq4CIHIB7h-O8iZc4DCZ6ya7ybc10X3FujWzQbbLtlhblq-NtjQnlUuD_BkuhokQcMbqo6pbY
2025-04-05 22:23:25,901 WARNING Retrying (Retry(total=9, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)': /upload/storage/v1/b/kaggle-competitions-submissions/o?uploadType=resumable&upload_id=AKDAyIslD5pB0lN6p7F1UdonN_6zYcdkS1NteRpq4CIHIB7h-O8iZc4DCZ6ya7ybc10X3FujWzQbbLtlhblq-NtjQnlUuD_BkuhokQcMbqo6pbY
2025-04-05 22:23:25,901 WARNING Retrying (Retry(

Submitted submission_11.csv to el-hackathon-2025


KeyboardInterrupt: 

In [ ]:
submission_df

,ID,C1,C2,C3,C4,C5,C6,C7,C8,C9,...,C26,C27,C28,C29,C30,C31,C32,C33,C34,C35
0,0,17.0,27.0,25.0,11.0,21.0,14.0,20.0,10.0,15.0,...,15.0,14.0,11.0,10.0,13.0,25.0,25.0,11.0,13.0,22.0
1,1,29.0,14.0,27.0,17.0,32.0,15.0,29.0,7.0,21.0,...,10.0,17.0,7.0,14.0,4.0,26.0,24.0,9.0,12.0,21.0
2,2,24.0,29.0,27.0,19.0,30.0,12.0,21.0,26.0,16.0,...,12.0,12.0,11.0,13.0,8.0,28.0,25.0,10.0,13.0,21.0
3,3,30.0,27.0,28.0,20.0,28.0,20.0,19.0,9.0,20.0,...,11.0,20.0,11.0,12.0,6.0,25.0,23.0,10.0,14.0,21.0
4,4,18.0,12.0,26.0,16.0,34.0,30.0,24.0,10.0,17.0,...,10.0,12.0,6.0,25.0,5.0,28.0,24.0,9.0,12.0,21.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2083,2083,29.0,26.0,28.0,17.0,23.0,24.0,17.0,10.0,25.0,...,12.0,19.0,14.0,10.0,8.0,25.0,23.0,8.0,16.0,22.0
2084,2084,27.0,27.0,26.0,16.0,26.0,19.0,19.0,12.0,21.0,...,13.0,10.0,11.0,11.0,9.0,25.0,25.0,9.0,13.0,21.0
2085,2085,24.0,28.0,25.0,14.0,24.0,13.0,21.0,10.0,17.0,...,13.0,16.0,12.0,16.0,11.0,26.0,25.0,12.0,20.0,21.0
2086,2086,29.0,26.0,28.0,21.0,8.0,19.0,20.0,9.0,21.0,...,11.0,20.0,12.0,12.0,4.0,25.0,24.0,10.0,14.0,21.0


# 4. Submission File Generation

Finally, we generate the submission file in the required format.  
Each row corresponds to a test spot with its identifier (constructed here as the index) followed by 35 predictions.

In [ ]:
!touch ~/.config/kaggle/kaggle.json
!chmod 600 ~/.config/kaggle/kaggle.json

'touch' is not recognized as an internal or external command,
operable program or batch file.
'chmod' is not recognized as an internal or external command,
operable program or batch file.
